# What is my pulse *actually* generating?

Every other demo asks "how far did the realized dynamics drift from the target?" and answers
with a fidelity. A fidelity tells you **that** something is wrong. It does not tell you
**what**.

`magnus_pauli` answers the second question. It takes a time-dependent $H(t)$ over an interval
and returns the effective generator, decomposed in the Pauli basis:

$$ U(T) = \exp\big(\Omega_1 + \Omega_2 + \dots\big), \qquad
\Omega_1 = -i\!\int_0^T\! H\,dt, \qquad
\Omega_2 = -\tfrac12\!\int_0^T\!\!dt_1\!\!\int_0^{t_1}\!\!dt_2\,[H(t_1), H(t_2)] $$

then writes each $\Omega_k$ as $\sum_P c_P\,P$ over Pauli strings $P \in \{I,X,Y,Z\}^{\otimes n}$.

The payoff is that **$\Omega_2$ contains interactions that appear in neither your intended gate
nor your drive terms** — they are manufactured by the non-commutation of $H$ with itself at
different times. Those are invisible to a fidelity number and obvious in a Pauli table.

Only orders 1 and 2 exist here, and that is deliberate rather than a shortcut: for a
spin-dependent force the series *terminates* at second order (see `05_ms_two_qubit_gate`), and
asking for order 3 raises rather than handing back a truncation you might mistake for exact.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm

import htdse as ht
from htdse.submodules.spin import sigma_x, sigma_z, pauli_term

def show(terms, top=8):
    """Print Magnus orders as Pauli tables, largest coefficient first."""
    for k, d in enumerate(terms, 1):
        items = sorted(d.items(), key=lambda kv: -abs(kv[1]))[:top]
        print(f"  Omega_{k}:")
        for p, c in items:
            print(f"    {p}   {c.imag:+.5f}i" if abs(c.real) < 1e-12
                  else f"    {p}   {c:+.5f}")

## 1. Sanity: a Hamiltonian that commutes with itself

If $H$ is constant, $[H(t_1),H(t_2)] = 0$, so $\Omega_1 = -iHT$ **exactly** and $\Omega_2$
vanishes. Anything else would mean the quadrature is wrong.

In [ ]:
H_const = pauli_term("X0", coeff=0.3) + pauli_term("Z1", coeff=0.2)
O1, O2 = ht.magnus(H_const, 1.7)

print("constant H, T = 1.7:")
show(ht.magnus_pauli(H_const, 1.7))
print(f"\n  Omega_1 == -i H T : {np.allclose(O1, -1j*np.asarray(H_const.hamiltonian(0))*1.7)}")
print(f"  Omega_2 == 0      : {np.max(np.abs(O2)) < 1e-12}")

## 2. The real question: an intended ZZ gate, driven imperfectly

Intended: a maximally entangling $ZZ$ rotation, $J\,\sigma_z\sigma_z$ run for $T = \pi/4J$, so
the intended generator is exactly $-i\frac{\pi}{4}ZZ$.

Reality: the drive that implements it leaks a small oscillating $X$ onto both qubits (an
off-resonant carrier, say). Nothing about that leak is a $ZZ$ error, so a fidelity would just
report "0.98" and leave you guessing.

Watch what the Pauli table says instead.

In [ ]:
J = 0.6
T = np.pi / (4 * J)          # intended: exp(-i pi/4 ZZ)
leak, w_leak = 0.25, 3.0     # amplitude and frequency of the stray X drive

H_real = (pauli_term("Z0Z1", coeff=J)
          + pauli_term("X0", coeff=lambda t: leak * np.cos(w_leak * t))
          + pauli_term("X1", coeff=lambda t: leak * np.cos(w_leak * t)))

print(f"intended generator: -i*pi/4 * ZZ  =  {-np.pi/4:+.5f}i ZZ\n")
print("what the pulse actually generates:")
show(ht.magnus_pauli(H_real, T))

Read the table:

- **`ZZ` at first order** sits at $-i\pi/4$ — the gate you asked for is there, essentially intact.
- **`IX`, `XI` at first order** — the leak, straightforwardly time-averaged. Unsurprising.
- **`YZ`, `ZY` at second order** — these are the interesting ones. There is no $YZ$ term
  anywhere in `H_real`. It exists purely because $ZZ$ and $X$ do not commute, so the
  *ordering* of the drive manufactures a correlated spin-flip error out of two things that
  individually look harmless.

That last row is the entire reason this tool exists. A fidelity would have hidden it inside one
number; here it is named, signed, and sized.

## 3. Does the expansion actually reproduce the dynamics?

Second order should beat first order against a real solve. If it does not, the expansion is
being pushed past where it converges and none of the table above should be trusted.

In [ ]:
with ht.quiet():
    U_true = np.asarray(ht.UnitaryEvolution(H_real, dim=4).unitary_at(T))
O1, O2 = ht.magnus(H_real, T)

F1 = ht.process_fidelity(expm(O1), U_true)
F2 = ht.process_fidelity(expm(O1 + O2), U_true)
print(f"  order 1 only : 1 - F = {1-F1:.3e}")
print(f"  orders 1 + 2 : 1 - F = {1-F2:.3e}   <- must be smaller")
print(f"  improvement  : {(1-F1)/(1-F2):.0f}x")

## 4. Using it: watch an unwanted term grow

The table is most useful swept. Here: how does the manufactured `YZ` coupling scale with the
leak amplitude, and how does that compare to the intended `ZZ`?

The `ZZ` line stays flat (the gate is doing its job); `YZ` grows quadratically, because it is a
second-order product of two first-order errors. That slope is the sort of thing you would
otherwise have to infer from a fidelity curve.

In [ ]:
leaks = np.linspace(0.0, 0.6, 13)
zz, yz = [], []
for L in leaks:
    H_L = (pauli_term("Z0Z1", coeff=J)
           + pauli_term("X0", coeff=lambda t, L=L: L * np.cos(w_leak * t))
           + pauli_term("X1", coeff=lambda t, L=L: L * np.cos(w_leak * t)))
    t1, t2 = ht.magnus_pauli(H_L, T)
    zz.append(abs(t1.get("ZZ", 0)))
    yz.append(abs(t2.get("YZ", 0)))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(leaks, zz, "o-", label="|ZZ| (intended, order 1)")
ax.plot(leaks, yz, "s-", label="|YZ| (manufactured, order 2)")
ax.plot(leaks, yz[-1] * (leaks / leaks[-1]) ** 2, "k--", lw=1, label="quadratic guide")
ax.set_xlabel("leak amplitude"); ax.set_ylabel("|Pauli coefficient|")
ax.set_title("an error that exists only because the terms don't commute")
ax.legend(); plt.tight_layout(); plt.show()

## Notes

- Coefficients are those of $\Omega$ itself, which is **anti-Hermitian** ($\Omega = -iH_{\rm eff}$),
  so they come out imaginary. Multiply by $i$ to read them as an effective Hamiltonian.
- `magnus_pauli` needs a pure qubit register ($\dim = 2^n$). A mechanism carrying a motional
  mode has to be reduced to the spins first — trace it out, or evaluate on a spin-only model.
- Quadrature is trapezoid on a uniform grid (`n_grid`); raise it if $H$ oscillates fast on the
  scale of $T/n_{\rm grid}$.
- The expansion assumes it converges. Section 3's check is not decoration — a small
  `1 - F` at second order is what licenses reading the table at all.